# Figure 3: database transfer, pooling and quantitative agreement

Figure 3 uses the 58 healthy-stool acquisitions in five sampling series. It shows how much of an
acquisition's peptide identification is kept when its database is selected from another
acquisition (b, c), what pooling the databases of 12 technical replicates adds (d, e), and how
closely protein-group and KEGG-ortholog intensities agree between acquisition pairs (f).
Panel a is a schematic. Panels b-e use 9-50-residue peptides (v30).

Run from the repository root: `python paper.py build Fig3`. Set `FASTALAKE_FONT_DIR` to the Söhne
fonts (`soehne-buch.otf`, `soehne-buch.ttf`, `soehne-halbfett.ttf`) for the pixel checks.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

import json
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 160)
d = panels.load(SD)
print({k: v.shape for k, v in d.items()})

## Panel a

Schematic of the five healthy-stool sampling series (artwork, not regenerated). Supports
"We next used the 58 healthy-stool acquisitions, which form five sampling series" (v28d line 99).
n: 58 acquisitions (12, 15, 10, 11 and 10 per series). No test. Every typed number is traced in
`ARTWORK_NUMBERS.md`; the counts below are recomputed from the transfer table.

In [ ]:
display(Image(filename=str(panels.preview(panels.render_artwork(UNIT / "artwork" / "panel_a_design.pdf", OUT / "panel_a.png")))))
print(panels.stat_a_design_counts(d["comparisons"]))


## Panel b

Peptide count with a transferred database as a percentage of the own-database count, for a
same-series donor (T1, blue; legend "Same series") and a different-donor database (T4, orange;
legend "Different donor"). Supports "retained a median 96.0% of the 9–50-residue peptide count ...
retaining 55.8% of the peptide count" (v30 paragraph 48; v28d line 99 printed the 5-50 values).
n: 58 target acquisitions (12, 15, 10, 11, 10 per series), one T1 and one T4 search each.
Descriptive medians; no test. The original producer drew b, c, d and e on one canvas, so they are
rendered together here; the legend relabel is a composite step (`assemble_v30`, final cell).


In [ ]:
res_bcde = panels.panels_bcde(d["comparisons"], OUT / "panel_bcde")
display(Image(filename=str(panels.preview(OUT / "panel_bcde.png"))))
bc = panels.stat_bc_transfer(d["comparisons"])
print(bc[bc.field.eq("count_retention_percent")].to_string(index=False))


## Panel c

Recovery of the own-database peptide identities with the transferred database. Supports "89.6% of
the peptide identities ... 50.2% of peptide identities" (v30 paragraph 48; v28d line 99). n as in
panel b. Descriptive medians; no test.


In [ ]:
print(bc[bc.field.eq("own_set_recovery_percent")].to_string(index=False))


## Panel d

Peptide count of each technical replicate searched against its own database, the union of all 12
replicate databases and the union of the other 11, as a percentage of the own-database count.
Supports "increased 9–50-residue peptide identifications in every acquisition by a median 10.0%"
and "(+10.7%, 97.5% retained)" (v30 paragraph 50; v28d line 103). The gained and lost medians
printed in v28d line 103 are no longer printed in v30. n: 12 acquisitions (repeated injections
of one digest). Descriptive medians; no test.


In [ ]:
de = panels.stat_de_pool(d["comparisons"])
for arm, v in de.items():
    print(arm, "count", v["count"], "gained", v["median_gained"], "lost", v["median_lost"],
          "acquisitions with more peptides", v["acquisitions_with_more_peptides"], "of", v["n"])


## Panel e

Recovery of the own-database peptide identities with the pooled databases. Supports "while
retaining 96.8% of identities from the acquisition-specific search" (v30 paragraph 50; v28d
line 103). n: 12 acquisitions. Descriptive medians; no test.


In [ ]:
for arm, v in de.items():
    print(arm, "identity", v["identity"])

## Panel f

Pairwise agreement within each sampling series. Upper row: protein groups (host pairs above the
diagonal, microbial below). Lower row: KEGG orthologs (all origins above, human-assigned signal
excluded below). Labels: median Pearson r of log2 intensities, median raw-intensity CV and median
number of jointly quantified features across pairs. Supports v28d line 109 / v30 paragraph 54
("21% ... 52%", "0.82 to 0.48", "0.90 ... 0.81", "more strongly correlated ... in every sampling
design") and the legend. Unchanged in v30 (no 9-50 counterpart in its producers). n: 66, 105, 45,
55 and 45 acquisition pairs from 12, 15, 10, 11 and 10 acquisitions; pairs share acquisitions and
are not independent. Descriptive medians; no test.


In [ ]:
res_f = panels.panel_f(d["pg"], d["ko"], d["design"], OUT / "panel_f")
display(Image(filename=str(panels.preview(OUT / "panel_f.png"))))
receipt = json.loads((UNIT / "expected" / "panel_f_producer_receipt.json").read_text())
print("against the producer receipt:", panels.check_panel_f_receipt(res_f, receipt))
fs = panels.stat_f_summaries(d["pg"], d["ko"])
print(fs.to_string(index=False))
print(panels.stat_f_ko_vs_microbial_pg(fs).to_string(index=False))


## Checks and numbers

Pixel checks compare each regenerated panel with the panel rendered by the original producer
(`expected/`), at 300 dpi: b-e against the v30 (9-50) producer render, f against the v28f
producer render (f is unchanged in v30). Band checks place each panel where the assembly chain
placed it and compare that band with the v30 vector composite (`expected/Fig3_v30_vector.pdf`);
they differ where the chain re-typeset text (`ASSEMBLY.md`) and are recorded as visual checks.
The composite check rebuilds the whole v30 page with `assemble_v30` and compares it with the
v30 figure PNG (`expected/Fig3_v30.png`). Then every value in `NUMBERS.tsv` is printed, followed by the donor assignments.


In [ ]:
from tools.check_pixels import compare
CHK = UNIT / "checks"
if panels.fonts_present():
    for x in ["bcde", "f"]:
        r = compare(OUT / f"panel_{x}.pdf", UNIT / "expected" / f"panel_{x}.pdf", CHK / f"panel_{x}", dpi=300)
        print(f"panel {x}:", r["verdict"], r.get("differing_pixels"))
    rects = panels.composite_rects()
    for x, src in [("a", UNIT / "artwork" / "panel_a_design.pdf"), ("bcde", OUT / "panel_bcde.pdf"), ("f", OUT / "panel_f.pdf")]:
        r = panels.composite_band_check(src, UNIT / "expected" / "Fig3_v30_vector.pdf", rects[x],
                                        CHK / f"composite_band_{x}", OUT / "bands")
        print(f"v30 band {x}:", r["verdict"], r.get("differing_pixels"), r.get("differing_fraction"))
    comp = panels.assemble_v30(OUT / "panel_bcde.pdf", UNIT / "expected" / "Fig3_v28f_relabel_base.pdf", OUT / "Fig3_v30")
    r = compare(comp["png"], UNIT / "expected" / "Fig3_v30.png", CHK / "composite_v30", dpi=300)
    print("v30 composite:", r["verdict"], r.get("differing_pixels"), "retyped spans", comp["retyped_spans"])
    v38 = panels.relabel_v38(comp["pdf"], OUT / "Fig3_v38")   # step 17: KO labels "<n> other" -> "<n> non-human"
    r = compare(v38["png"], UNIT / "expected" / "Fig3_v38.png", CHK / "composite_v38", dpi=300)
    print("v38 composite:", r["verdict"], r.get("differing_pixels"), v38["labels"])
    latest = panels.relabel_v42(v38["pdf"], OUT / "Fig3_v42")
    current = compare(latest["png"], UNIT / "expected" / "Fig3_v42.png", CHK / "composite_v42", dpi=300)
    print("v42 composite:", current["verdict"], current.get("differing_pixels"))
    assert current["verdict"] == "IDENTICAL", current
else:
    print("pixel checks SKIPPED: fonts absent")

for key, value in panels.numbers(d):
    print(f"{key}\t{value}")
print(panels.stat_methods_donor_assignments(d["comparisons"]))

import csv
from tools.compare_values import same_numeric_value
registered = list(csv.DictReader((UNIT / "NUMBERS.tsv").open(), delimiter="\t"))
for index in (22, 23):
    key, value = panels.numbers(d)[index]
    assert same_numeric_value(value, registered[index]["v42_recomputed_value"]), (key, value)
print("v42 human-excluded KO values match the registered source rows")


In [ ]:
# Check the current human-excluded KO statistical records.
import csv
from pathlib import Path
stats_path = Path(panels.__file__).with_name("STATS.tsv")
records = {r["claim"]: r for r in csv.DictReader(stats_path.open(), delimiter="\t")}
for design, scope in [("TechnicalReplicates", "technical replicates"), ("Interindividual", "individuals")]:
    claim = f"KO profiles (human-assigned signal excluded): median pairwise r between {scope}"
    row = fs[(fs.space == "KO") & (fs.design == design) & (fs.cls == "new_other_unresolved")].iloc[0]
    assert records[claim]["estimate"] == f"median Pearson r {row.median_r:.4f}"
    assert records[claim]["uncertainty"] == f"r IQR {row.r_q1:.4f} to {row.r_q3:.4f}"
print("Current KO statistical records reproduced")
